# 00 Environment and Foundry Agents

Validate package dependencies, Azure ML defaults, and the deployed Foundry agents in `rg-microsoft-iq`.


## 1) Notebook Dependency Bootstrap

Validate the packages this notebook needs before importing the pipeline dependencies.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "azure-ai-ml": "azure.ai.ml",
    "azure-ai-projects": "azure.ai.projects",
    "azure-identity": "azure.identity",
    "python-dotenv": "dotenv",
    "kaggle": "kaggle",
    "mltable": "mltable",
}

def module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

missing_packages = [
    package_name
    for package_name, module_name in REQUIRED_PACKAGES.items()
    if not module_available(module_name)
]

if missing_packages:
    print("Installing missing packages with the active notebook interpreter:", missing_packages)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *missing_packages,
    ])
    importlib.invalidate_caches()
else:
    print("All required packages are already available in this notebook environment.")

print("Python executable:", sys.executable)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import json
import os
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.datasets import fetch_openml

candidate_src_paths = [
    Path.cwd() / "src",
    Path.cwd().parent / "aml-foundry-integration" / "src",
    Path.cwd().parent.parent / "aml-foundry-integration" / "src",
]
for package_src in candidate_src_paths:
    if (package_src / "aml_bandits" / "settings.py").exists():
        if str(package_src) not in sys.path:
            sys.path.insert(0, str(package_src))
        break

RNG = np.random.default_rng(42)
pd.set_option("display.max_columns", 120)


## 2) Azure ML Configuration

Use environment variables first. The resource group and workspace defaults match this local demo workspace and do not include secrets.

In [ ]:
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.identity import DefaultAzureCredential

from aml_bandits.settings import (
    apply_project_environment_defaults,
    load_project_defaults,
    project_summary,
)

load_dotenv()
PROJECT_DEFAULTS = apply_project_environment_defaults(load_project_defaults())
PROJECT_BINDING = project_summary(PROJECT_DEFAULTS)

DEFAULT_AZUREML_RESOURCE_GROUP = PROJECT_DEFAULTS.azure_ml.resource_group
DEFAULT_AZUREML_WORKSPACE_NAME = PROJECT_DEFAULTS.azure_ml.workspace_name
DEFAULT_AZUREML_LOCATION = PROJECT_DEFAULTS.azure_ml.location

AZUREML_SUBSCRIPTION_ID = (
    os.getenv("AZUREML_SUBSCRIPTION_ID")
    or os.getenv("AZURE_SUBSCRIPTION_ID")
    or os.getenv("SUBSCRIPTION_ID")
    or ""
)
AZUREML_RESOURCE_GROUP = os.getenv("AZUREML_RESOURCE_GROUP", DEFAULT_AZUREML_RESOURCE_GROUP)
AZUREML_WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", DEFAULT_AZUREML_WORKSPACE_NAME)
AZUREML_LOCATION = os.getenv("AZUREML_LOCATION", DEFAULT_AZUREML_LOCATION)
AZUREML_DATA_ASSET_NAME = os.getenv("AZUREML_DATA_ASSET_NAME", "bank-marketing-7mlet")
AZUREML_DATA_ASSET_VERSION = os.getenv("AZUREML_DATA_ASSET_VERSION", "1")

INGESTION_DIR = Path(
    os.getenv(
        "BANK_MARKETING_INGESTION_DIR",
        str(Path.cwd() / "tmp" / "7mlet" / "azureml_ingestion"),
    )
).resolve()
INGESTION_DIR.mkdir(parents=True, exist_ok=True)
os.environ["BANK_MARKETING_INGESTION_DIR"] = str(INGESTION_DIR)

AZUREML_SPEC_DIR = Path(
    os.getenv(
        "AZUREML_SPEC_DIR",
        str(Path.cwd() / "tmp" / "7mlet" / "azureml_specs"),
    )
).resolve()
AZUREML_SPEC_DIR.mkdir(parents=True, exist_ok=True)

print("Azure ML resource group:", AZUREML_RESOURCE_GROUP)
print("Azure ML workspace:", AZUREML_WORKSPACE_NAME)
print("Azure ML location:", AZUREML_LOCATION)
print("Subscription configured:", bool(AZUREML_SUBSCRIPTION_ID))
print("Foundry project endpoint:", os.getenv("FOUNDRY_PROJECT_ENDPOINT"))
print("Foundry feature agent:", os.getenv("FOUNDRY_FEATURE_AGENT_NAME"))
print("Foundry strategy agent:", os.getenv("FOUNDRY_STRATEGY_AGENT_NAME"))
print("Available Foundry agents:", os.getenv("FOUNDRY_AVAILABLE_AGENTS"))
print("Local ingestion folder:", INGESTION_DIR)
print("Azure ML spec folder:", AZUREML_SPEC_DIR)


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
client = AIProjectClient(
    endpoint=endpoint,
    credential=DefaultAzureCredential(exclude_interactive_browser_credential=False),
)
try:
    deployed_agents = list(client.agents.list_agents())
except AttributeError:
    deployed_agents = list(client.agents.list())

agent_inventory = pd.DataFrame([
    {
        "id": getattr(agent, "id", None),
        "name": getattr(agent, "name", None),
        "model": getattr(agent, "model", None),
        "description": getattr(agent, "description", None),
    }
    for agent in deployed_agents
])
print("Default workload agent:", os.getenv("FOUNDRY_STRATEGY_AGENT_NAME"))
agent_inventory
